In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import h5py
import anndata as ad
import matplotlib.pyplot as plt
import warnings
import os
import squidpy as sq
import spatialdata as sd
import spatialdata_io as sio
import spatialdata_plot

/opt/anaconda3/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [9]:
import tarfile
from pathlib import Path

from xenium_gse250346_loader import load_via_prebuilt_matrix, build_percell_counts_from_transcripts, load_cell_centroids


# The archive sometimes extracts to a nested folder (e.g. "outs/") —
# find wherever cells.csv.gz actually landed
outs_dir = Path("data/GSM7990532_output-XETG00048__0003392__THD0008__20230313__191400/outs")
if not (outs_dir / "cells.csv.gz").exists():
    matches = list(outs_dir.rglob("cells.csv.gz"))
    if not matches:
        raise FileNotFoundError(f"No cells.csv.gz found under {outs_dir}")
    outs_dir = matches[0].parent

# Fast path: prebuilt cell x gene matrix
adata = load_via_prebuilt_matrix(outs_dir)
print(adata)

# Or manual path: rebuild from raw transcripts.csv.gz with your own QC
counts = build_percell_counts_from_transcripts(outs_dir, qv_threshold=20.0)
centroids = load_cell_centroids(outs_dir)
counts = counts.join(centroids, how="inner")
print(counts.shape)

/opt/anaconda3/lib/python3.12/site-packages/spatialdata/_core/spatialdata.py:170: UserWarning: The table is annotating 'cell_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)


AnnData object with n_obs × n_vars = 73785 × 343
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region', 'z_level'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'
Streaming transcripts.csv.gz in chunks of 2,000,000 rows ...
  processed chunk 8
Pivoting to per-cell x gene matrix ...
(73783, 346)
